In [1]:
# Install the Required Packages
!pip install -q -U bitsandbytes --use-deprecated=legacy-resolver
!pip install -q -U git+https://github.com/huggingface/transformers.git --use-deprecated=legacy-resolver
!pip install -q -U git+https://github.com/huggingface/peft.git --use-deprecated=legacy-resolver
!pip install -q -U git+https://github.com/huggingface/accelerate.git --use-deprecated=legacy-resolver
!pip install -q -U datasets scipy ipywidgets matplotlib --use-deprecated=legacy-resolver
!pip install accelerate
!pip install boto3

In [ ]:
import os
import boto3
from botocore.exceptions import NoCredentialsError

aws_access_key_id = os.environ['AWS_ACCESS_KEY_ID']  # AWS S3 bucket Access Key ID
aws_secret_access_key = os.environ['AWS_SECRET_ACCESS_KEY'] # AWS S3 bucket Secret Access Key
aws_region = 'us-east-1'

# Create a session using my AWS credentials
session = boto3.Session(
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    region_name=aws_region
)

In [3]:
# Define my S3 bucket name and folder
bucket_name = 'archistackoverflow'
s3_folder = ''

# Create an S3 client
s3 = session.client('s3')

# List files in the specified my S3 folder
response = s3.list_objects_v2(Bucket=bucket_name, Prefix=s3_folder)

if 'Contents' in response:
    files = [content['Key'] for content in response['Contents']]
    print("Files in S3 bucket:")
    for file in files:
        print(file)
else:
    print("No files found in the S3 bucket or folder.")

Files in S3 bucket:
eval.jsonl
test.jsonl
train.jsonl


In [4]:
import os

# Create a local directory inside my colab drive to store downloaded files
local_dir = '/home/ec2-user/SageMaker'
os.makedirs(local_dir, exist_ok=True)

# Download each file from S3
for file_key in files:
    local_file_path = os.path.join(local_dir, os.path.basename(file_key))
    try:
        s3.download_file(bucket_name, file_key, local_file_path)
        print(f"Downloaded {file_key} to {local_file_path}")
    except NoCredentialsError:
        print("Credentials not available")
    except FileNotFoundError:
        print(f"The file {file_key} was not found in S3")
%cd /home/ec2-user/SageMaker

Downloaded eval.jsonl to /home/ec2-user/SageMaker/eval.jsonl
Downloaded test.jsonl to /home/ec2-user/SageMaker/test.jsonl
Downloaded train.jsonl to /home/ec2-user/SageMaker/train.jsonl
/home/ec2-user/SageMaker


In [5]:
from datasets import load_dataset

bot_train_dataset = load_dataset('json', data_files='train.jsonl', split='train')
bot_eval_dataset = load_dataset('json', data_files='eval.jsonl', split='train')

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/importlib/__init__.py:126: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 1.22.4)
  return _bootstrap._gcd_import(name[level:], package, level)


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [6]:
def formatting_func(example):
    text = f"### Question: {example['prompt']}\n ### Answer: {example['response']}"   ## Here prompt and response are keys in jsonl files
    return text

In [ ]:
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])  # My Access Token Hugging Face
from huggingface_hub import notebook_login

notebook_login()

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

base_model_id = "mistralai/Mistral-7B-v0.1"   # My Base Model For Train
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(base_model_id, device_map="auto")

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/torch/cuda/__init__.py:654: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [9]:
tokenizer = AutoTokenizer.from_pretrained(
    base_model_id,
    padding_side="left",
    add_eos_token=True,
    add_bos_token=True,
)
tokenizer.pad_token = tokenizer.eos_token

def generate_and_tokenize_prompt(prompt):
    return tokenizer(formatting_func(prompt))

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [10]:
tokenized_train_dataset = bot_train_dataset.map(generate_and_tokenize_prompt)
tokenized_val_dataset = bot_eval_dataset.map(generate_and_tokenize_prompt)

Map:   0%|          | 0/350083 [00:00<?, ? examples/s]

Map:   0%|          | 0/43760 [00:00<?, ? examples/s]

In [14]:
max_length = 512 # This was an appropriate max length for my dataset

def generate_and_tokenize_prompt2(prompt):
    result = tokenizer(
        formatting_func(prompt),
        truncation=True,
        max_length=max_length,
        padding="max_length",
    )
    result["labels"] = result["input_ids"].copy()
    return result

In [15]:
tokenized_train_dataset = bot_train_dataset.map(generate_and_tokenize_prompt2)
tokenized_val_dataset = bot_eval_dataset.map(generate_and_tokenize_prompt2)

Map:   0%|          | 0/350083 [00:00<?, ? examples/s]

Map:   0%|          | 0/43760 [00:00<?, ? examples/s]

In [16]:
from peft import prepare_model_for_kbit_training

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

In [17]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

In [18]:
print(model)

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralSdpaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): MistralRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
     

In [19]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
        "lm_head",
    ],
    bias="none",
    lora_dropout=0.05,  # Conventional
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, config)
print_trainable_parameters(model)

The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.


trainable params: 85041152 || all params: 7326773248 || trainable%: 1.1606903765339511


In [20]:
print(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32000, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralSdpaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): 

In [21]:
if torch.cuda.device_count() > 1: # If more than 1 GPU for parellel training
    model.is_parallelizable = True
    model.model_parallel = True

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [22]:
from accelerate import Accelerator

# Initialize the accelerator
accelerator = Accelerator()

In [23]:
model = accelerator.prepare_model(model)

In [ ]:
import transformers
from datetime import datetime

project = "stackoverflow-finetune"
base_model_name = "mistral"
run_name = base_model_name + "-" + project
output_dir = "./" + run_name

trainer = transformers.Trainer(
    model=model,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    args=transformers.TrainingArguments(
        output_dir=output_dir,
        warmup_steps=2,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=1,
        gradient_checkpointing=True,
        max_steps=500,
        learning_rate=2.5e-5, # Want a small lr for finetuning
        bf16=True,
        optim="paged_adamw_8bit",
        logging_steps=25,              # When to start reporting loss
        logging_dir="./logs",        # Directory for storing logs
        save_strategy="steps",       # Save the model checkpoint every logging step
        save_steps=25,                # Save checkpoints every 50 steps
        evaluation_strategy="steps", # Evaluate the model every logging step
        eval_steps=25,               # Evaluate and save checkpoints every 50 steps
        do_eval=True,                # Perform evaluation at the end of training
      #  report_to="wandb",           # Comment this out if you don't want to use weights & baises
        run_name=f"{run_name}-{datetime.now().strftime('%Y-%m-%d-%H-%M')}"          # Name of the W&B run (optional)
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

model.config.use_cache = False  # silence the warnings. Please re-enable for inference!
trainer.train()

In [ ]:
def upload_directory(directory_path, bucket_name, s3_path):
    for root, dirs, files in os.walk(directory_path):
        for filename in files:
            local_path = os.path.join(root, filename)
            relative_path = os.path.relpath(local_path, directory_path)
            s3_file_path = os.path.join(s3_path, relative_path)
            try:
                s3.upload_file(local_path, bucket_name, s3_file_path)
                print(f'Successfully uploaded {local_path} to s3://{bucket_name}/{s3_file_path}')
            except FileNotFoundError:
                print(f'The file {local_path} was not found')
            except NoCredentialsError:
                print('Credentials not available')

# Define your checkpoint folder and S3 bucket details
checkpoint_folder = '/home/ec2-user/SageMaker/checkpoint-500'
bucket_name = 'archistackoverflow'
s3_folder_path = '.'

# Upload the checkpoint folder to S3
upload_directory(checkpoint_folder, bucket_name, s3_folder_path)